# Qwen2.5-3B-Instruct GPTQ-Int4 — Kaggle T4x2 Serving
## OpenAI-compatible REST API · cloudflared public tunnel · optional API key

| Setting | Value |
|---------|-------|
| Model | `Qwen/Qwen2.5-3B-Instruct-GPTQ-Int4` |
| Quantization | GPTQ Int4 (~2 GB vs ~6 GB float16) |
| GPUs | 2 x T4 (16 GB each) via `device_map="auto"` |
| Endpoint | OpenAI `POST /v1/chat/completions` |
| Tunnel | cloudflare quick tunnel — no account needed |
| Auth | Optional `Authorization: Bearer <key>` |

> **Why GPTQ-Int4?** The official Qwen GPTQ checkpoint is pre-calibrated —
> quality loss is minimal (~0.5 perplexity points) and the model loads 3x faster
> and uses 3x less VRAM than float16.
>
> **Swap model:** Change `MODEL_NAME` in Cell 2.
> Other options: `"Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4"`, `"Qwen/Qwen2.5-14B-Instruct-GPTQ-Int4"`.
>
> **Stop server:** run `cf_proc.terminate(); _server.should_exit = True`.


In [ ]:
# optimum provides GPTQ integration in transformers; auto-gptq does the kernel work.
# The CUDA 11.8 wheel is pre-built for Kaggle T4 GPUs.
!pip install -q optimum
!pip install -q auto-gptq --extra-index-url https://huggingface.github.io/autogptq-index/whl/cu118/
!pip install -q fastapi "uvicorn[standard]" pydantic nest_asyncio accelerate


In [ ]:
# ── Edit these before running ────────────────────────────────────────────────

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct-GPTQ-Int4"
# Other pre-quantized options (all from Qwen team):
#   "Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4"   — 7B fits on 2x T4 comfortably
#   "Qwen/Qwen2.5-14B-Instruct-GPTQ-Int4"  — 14B, tight on 2x T4

PORT = 8000

# Leave empty "" for open access, or set a secret string to require Bearer auth.
API_KEY = ""   # e.g. "my-secret-42"

MAX_NEW_TOKENS_DEFAULT = 512


In [ ]:
import warnings; warnings.filterwarnings("ignore")
import os, re, time, subprocess, threading, asyncio, uuid
import torch
from datetime import datetime

import nest_asyncio
nest_asyncio.apply()   # lets uvicorn run inside Jupyter's event loop

print("=" * 60)
print("ENVIRONMENT")
print("=" * 60)
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.version.cuda}")
n_gpu = torch.cuda.device_count()
print(f"GPUs     : {n_gpu}")
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"  cuda:{i}  {p.name}  {p.total_memory/1e9:.1f} GB")
print(f"model    : {MODEL_NAME}")
print(f"auth     : {'bearer token required' if API_KEY else 'disabled (open access)'}")
print("=" * 60)

if n_gpu == 0:
    raise RuntimeError("No GPU found — enable GPU accelerator in Kaggle settings.")


In [ ]:
# Authenticate with HuggingFace via Kaggle Secret.
# Notebook -> Add-ons -> Secrets -> Add New Secret  (name: HF_TOKEN)
# Qwen2.5-3B-Instruct-GPTQ-Int4 is ungated; token avoids rate-limits only.

try:
    from kaggle_secrets import UserSecretsClient
    _tok = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = _tok
    from huggingface_hub import login
    login(token=_tok, add_to_git_credential=False)
    print("HuggingFace: authenticated via Kaggle Secret HF_TOKEN")
except Exception as _e:
    print(f"HuggingFace: no token ({_e}) — continuing unauthenticated")


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, GPTQConfig

print("Loading tokenizer ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"Loading GPTQ-Int4 model (first run downloads ~2 GB) ...")
_t0 = time.time()

# GPTQConfig tells transformers this is a pre-quantized GPTQ checkpoint.
# disable_exllama=False uses the fast ExLlama CUDA kernel (much faster on T4).
# bits=4 matches the Int4 checkpoint.
_gptq_cfg = GPTQConfig(bits=4, disable_exllama=False)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=_gptq_cfg,
    device_map="auto",        # splits layers across cuda:0 and cuda:1
    trust_remote_code=True,
)
model.eval()

# With device_map="auto", model.device is unreliable — use hf_device_map.
INPUT_DEVICE = next(iter(model.hf_device_map.values()))   # typically "cuda:0"

_elapsed = time.time() - _t0
_params  = sum(p.numel() for p in model.parameters()) / 1e9

print(f"\nLoaded in {_elapsed:.1f}s   |   {_params:.2f}B parameters")
print(f"input device : {INPUT_DEVICE}")
print(f"device map   : {dict(list(model.hf_device_map.items())[:6])} ...")
print()
for i in range(torch.cuda.device_count()):
    used  = torch.cuda.memory_allocated(i) / 1e9
    total = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"  cuda:{i} VRAM  {used:.2f} / {total:.1f} GB used")


In [ ]:
from fastapi import FastAPI, HTTPException, Depends, Security
from fastapi.security import HTTPBearer, HTTPAuthorizationCredentials
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional

app = FastAPI(title="Qwen GPTQ-Int4 API", version="1.0.0")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"], allow_methods=["*"], allow_headers=["*"],
)

# ── Auth ──────────────────────────────────────────────────────────────────────
_http_bearer = HTTPBearer(auto_error=False)

async def check_auth(creds: HTTPAuthorizationCredentials = Security(_http_bearer)):
    if not API_KEY:
        return
    if creds is None or creds.credentials != API_KEY:
        raise HTTPException(status_code=401, detail="Missing or invalid API key")

# ── Pydantic models ───────────────────────────────────────────────────────────
class Message(BaseModel):
    role: str
    content: str

class ChatRequest(BaseModel):
    model: str = MODEL_NAME
    messages: list[Message]
    max_tokens: int = MAX_NEW_TOKENS_DEFAULT
    temperature: float = 0.7
    top_p: float = 0.9
    do_sample: Optional[bool] = None

# ── Routes ────────────────────────────────────────────────────────────────────
@app.get("/health")
def health():
    return {
        "status": "healthy",
        "model": MODEL_NAME,
        "quantization": "GPTQ-Int4",
        "auth_required": bool(API_KEY),
        "gpus": [
            {
                "index": i,
                "name": torch.cuda.get_device_properties(i).name,
                "vram_used_gb":  round(torch.cuda.memory_allocated(i)  / 1e9, 2),
                "vram_total_gb": round(torch.cuda.get_device_properties(i).total_memory / 1e9, 1),
            }
            for i in range(torch.cuda.device_count())
        ],
        "ts": datetime.utcnow().isoformat() + "Z",
    }

@app.get("/v1/models", dependencies=[Depends(check_auth)])
def list_models():
    return {
        "object": "list",
        "data": [{"id": MODEL_NAME, "object": "model", "owned_by": "local"}],
    }

@app.post("/v1/chat/completions", dependencies=[Depends(check_auth)])
def chat(req: ChatRequest):
    # 1. Build prompt from chat template
    prompt = tokenizer.apply_chat_template(
        [{"role": m.role, "content": m.content} for m in req.messages],
        tokenize=False,
        add_generation_prompt=True,
    )

    # 2. Tokenise — send to INPUT_DEVICE (first layer from hf_device_map)
    enc = tokenizer(prompt, return_tensors="pt").to(INPUT_DEVICE)
    prompt_len = enc["input_ids"].shape[1]

    # 3. Generation kwargs
    temp      = float(req.temperature)
    do_sample = req.do_sample if req.do_sample is not None else (temp > 0)
    gen_kw = dict(
        input_ids=enc["input_ids"],
        attention_mask=enc["attention_mask"],
        max_new_tokens=req.max_tokens,
        top_p=float(req.top_p),
        do_sample=do_sample,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    if do_sample:
        gen_kw["temperature"] = temp

    # 4. Generate
    t0 = time.perf_counter()
    with torch.no_grad():
        out_ids = model.generate(**gen_kw)
    latency_ms = (time.perf_counter() - t0) * 1000

    # 5. Decode only the newly generated tokens
    new_ids = out_ids[0][prompt_len:]
    answer  = tokenizer.decode(new_ids, skip_special_tokens=True).strip()

    n_out = len(new_ids)
    return {
        "id": f"chatcmpl-{uuid.uuid4().hex[:12]}",
        "object": "chat.completion",
        "created": int(time.time()),
        "model": req.model,
        "choices": [{
            "index": 0,
            "message": {"role": "assistant", "content": answer},
            "finish_reason": "stop",
        }],
        "usage": {
            "prompt_tokens":     prompt_len,
            "completion_tokens": n_out,
            "total_tokens":      prompt_len + n_out,
            "latency_ms":        round(latency_ms, 1),
            "tokens_per_sec":    round(n_out / (latency_ms / 1000), 1),
        },
    }

print("FastAPI app ready.")
print("  GET  /health")
print("  GET  /v1/models")
print("  POST /v1/chat/completions")


In [ ]:
import uvicorn

# Start uvicorn via asyncio — nest_asyncio.apply() (Cell 3) makes this safe in Jupyter.
_loop = asyncio.get_event_loop()
_config = uvicorn.Config(app, host="0.0.0.0", port=PORT,
                          log_level="warning", loop="asyncio")
_server = uvicorn.Server(_config)

server_thread = threading.Thread(
    target=_loop.run_until_complete,
    args=(_server.serve(),),
    daemon=True,
)
server_thread.start()
time.sleep(3)
print(f"Server: http://localhost:{PORT}")

# Download cloudflared binary
CF_BIN = "/kaggle/working/cloudflared"
if not os.path.exists(CF_BIN):
    print("Downloading cloudflared ...")
    subprocess.run([
        "wget", "-q", "-O", CF_BIN,
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
    ], check=True)
    subprocess.run(["chmod", "+x", CF_BIN], check=True)

# Launch quick tunnel and capture public URL from stderr
cf_proc = subprocess.Popen(
    [CF_BIN, "tunnel", "--url", f"http://localhost:{PORT}"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.PIPE,
    text=True,
)

public_url = None
deadline = time.time() + 45
while time.time() < deadline:
    line = cf_proc.stderr.readline()
    if not line:
        time.sleep(0.2)
        continue
    hit = re.search(r"https://[a-zA-Z0-9-]+[.]trycloudflare[.]com", line)
    if hit:
        public_url = hit.group(0)
        break

if not public_url:
    try:
        remaining = cf_proc.stderr.read(2000)
        hit = re.search(r"https://[a-zA-Z0-9-]+[.]trycloudflare[.]com", remaining)
        if hit:
            public_url = hit.group(0)
    except Exception:
        pass

if public_url:
    print()
    print("=" * 60)
    print("PUBLIC URL:")
    print(f"  {public_url}")
    print()
    print(f"  GET  {public_url}/health")
    print(f"  GET  {public_url}/v1/models")
    print(f"  POST {public_url}/v1/chat/completions")
    if API_KEY:
        print(f"\n  Authorization: Bearer {API_KEY}")
    else:
        print("\n  No auth required")
    print("=" * 60)
else:
    print("WARNING: could not read cloudflare URL from stderr.")
    print("Try: cf_proc.stderr.read(2000)")


In [ ]:
import requests

url  = f"http://localhost:{PORT}/v1/chat/completions"
hdrs = {"Authorization": f"Bearer {API_KEY}"} if API_KEY else {}

payload = {
    "model": MODEL_NAME,
    "messages": [
        {"role": "system", "content": "You are a concise assistant."},
        {"role": "user",   "content": "What is 2 + 2? Answer in one word."},
    ],
    "max_tokens": 20,
    "temperature": 0.0,
}

resp = requests.post(url, json=payload, headers=hdrs, timeout=60)
resp.raise_for_status()
data = resp.json()

print("Answer     :", data["choices"][0]["message"]["content"])
print("Usage      :", data["usage"])


## Usage from outside Kaggle

Replace `PUBLIC_URL` with the URL printed by Cell 7.

### curl — no auth
```bash
curl -s -X POST PUBLIC_URL/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{
        "model": "Qwen/Qwen2.5-3B-Instruct-GPTQ-Int4",
        "messages": [{"role":"user","content":"Hello!"}],
        "max_tokens": 200,
        "temperature": 0.7
      }' | python -m json.tool
```

### curl — with API key
```bash
curl -s -X POST PUBLIC_URL/v1/chat/completions \
  -H "Content-Type: application/json" \
  -H "Authorization: Bearer my-secret-42" \
  -d '{"model":"Qwen/Qwen2.5-3B-Instruct-GPTQ-Int4","messages":[{"role":"user","content":"Hi"}],"max_tokens":100}' \
  | python -m json.tool
```

### Python openai SDK (drop-in compatible)
```python
from openai import OpenAI

client = OpenAI(
    base_url="PUBLIC_URL/v1",
    api_key="my-secret-42",   # any non-empty string when auth is disabled
)

resp = client.chat.completions.create(
    model="Qwen/Qwen2.5-3B-Instruct-GPTQ-Int4",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user",   "content": "Explain GPTQ quantization in two sentences."},
    ],
    max_tokens=300,
)
print(resp.choices[0].message.content)
```

### Stop the tunnel + server
```python
cf_proc.terminate()
_server.should_exit = True
```
